# 🏛️ Institutional-Style Multi-Factor Equity Screener

**A production-grade quantitative research platform for ranking S&P 500 equities on Value, Momentum, Quality, Volatility, and Size factors, constructing a long-only factor portfolio, and benchmarking it against the S&P 500.**

---

## 1. Executive Project Overview

This notebook implements a full **multi-factor equity screening system** in the style used by quantitative asset managers (e.g., AQR, BlackRock Factor Investing, MSCI Barra, Dimensional Fund Advisors). It automatically:

1. Pulls the current S&P 500 constituent list and historical price/fundamental data.
2. Cleans and aligns messy real-world financial data (missing statements, delisted tickers, NaNs).
3. Engineers five institutional factors: **Value, Momentum, Quality, Low-Volatility, and Size**.
4. Cross-sectionally standardizes each factor with **Z-scores** so that factors measured in different units (dollars, percentages, ratios) become comparable.
5. Combines the standardized factors into a single **Composite Factor Score** and ranks the entire universe.
6. Builds an **equal-weight long-only portfolio** from the top-ranked decile and rebalances it periodically.
7. Computes institutional **performance analytics** (CAGR, Sharpe, Sortino, max drawdown, alpha/beta, information ratio) versus the **SPY** benchmark.
8. Renders a set of **interactive Plotly dashboards** suitable for a portfolio tear sheet.

The notebook is written to double as a **teaching document**: every quantitative concept is explained in plain English with its formula *before* it is implemented in code, so it can be read end-to-end by a recruiter, a professor, or a fellow quant without needing outside references.

> **Note on scope:** Running the full S&P 500 (500 tickers) through `yfinance` fundamental lookups can take 10–20 minutes and is subject to Yahoo Finance rate limiting. A `UNIVERSE_SIZE` parameter lets you run a fast demo (e.g., 50–100 names) or the full universe.


## 2. Why This Project Matters in Institutional Finance

Multi-factor investing is the quantitative backbone of modern **factor-based / "smart beta"** asset management, which manages several trillion dollars globally. Institutions do not pick stocks on narrative alone — they decompose returns into systematic, repeatable drivers ("factors") that have historically earned a risk premium:

- **Value** — cheap stocks (low P/E, P/B, EV/EBITDA) have historically outperformed expensive ones over long horizons (Fama-French).
- **Momentum** — stocks that have recently outperformed tend to keep outperforming over 3–12 month horizons (Jegadeesh & Titman).
- **Quality** — profitable, low-leverage companies tend to compound value more reliably and survive downturns.
- **Low Volatility** — lower-risk stocks have historically delivered better risk-adjusted returns than theory would predict (the "low-vol anomaly").
- **Size** — smaller-capitalization companies have historically carried a return premium over large caps, compensating for liquidity and information risk.

This project demonstrates the entire pipeline an institutional quant researcher owns: **data engineering → factor research → portfolio construction → risk/performance attribution → presentation**, which is exactly the skill set tested in quant research and portfolio management interviews.


## 3. Real-World Use Cases

**Hedge Funds (Equity Long/Short, Quant Funds)**
- Factor screens generate the long book (top decile) and short book (bottom decile) of market-neutral strategies.
- Factor exposures are monitored to keep the fund "factor-neutral" against unwanted risk (e.g., unintended sector or beta bets).

**Asset Managers / Mutual Funds / ETFs**
- Smart-beta and factor ETFs (e.g., iShares MSCI USA Quality Factor, Vanguard Value) are built from screens structurally similar to this one.
- Factor tilts are used in strategic asset allocation and fund-of-fund manager selection.

**Investment Banks (Equity Research & Prime Brokerage)**
- Equity research desks use factor scores to contextualize single-stock recommendations against the broader peer universe.
- Prime brokerage quant strategy teams provide factor analytics to hedge fund clients as a value-added service.

**Quantitative Research Firms**
- Firms like MSCI Barra, Axioma, and Qontigo sell commercial multi-factor risk models (this notebook is a simplified, transparent version of that same architecture).
- Academic-style factor research (Fama-French, AQR) is the theoretical foundation being operationalized here.


## 4. Complete System Architecture

```
                       ┌────────────────────────┐
                       │   S&P 500 Universe      │
                       │  (Wikipedia constituent │
                       │        list)             │
                       └───────────┬────────────┘
                                   │
                    ┌──────────────▼───────────────┐
                    │      DATA COLLECTION LAYER     │
                    │  yfinance: prices, fundamentals│
                    │  (FMP / AlphaVantage / FRED     │
                    │   optional plug-ins)            │
                    └──────────────┬───────────────┘
                                   │
                    ┌──────────────▼───────────────┐
                    │      DATA CLEANING LAYER       │
                    │  missing data, NaNs, delistings │
                    └──────────────┬───────────────┘
                                   │
                    ┌──────────────▼───────────────┐
                    │   FEATURE ENGINEERING LAYER    │
                    │ Value / Momentum / Quality /   │
                    │ Volatility / Size raw metrics   │
                    └──────────────┬───────────────┘
                                   │
                    ┌──────────────▼───────────────┐
                    │   FACTOR SCORING LAYER         │
                    │  Cross-sectional Z-scores  →   │
                    │  Composite Score  →  Ranking    │
                    └──────────────┬───────────────┘
                                   │
                    ┌──────────────▼───────────────┐
                    │  PORTFOLIO CONSTRUCTION LAYER  │
                    │  Top-decile equal/score weight  │
                    │  Periodic rebalancing           │
                    └──────────────┬───────────────┘
                                   │
                    ┌──────────────▼───────────────┐
                    │  PERFORMANCE ANALYTICS LAYER   │
                    │  Returns, risk, benchmark vs SPY│
                    └──────────────┬───────────────┘
                                   │
                    ┌──────────────▼───────────────┐
                    │   VISUALIZATION / DASHBOARD    │
                    │   Plotly + Matplotlib tear sheet│
                    └────────────────────────────────┘
```


## 5. Required APIs and Data Sources

| Source | Used For | Required? |
|---|---|---|
| **Yahoo Finance** (`yfinance`) | Historical prices, market cap, valuation & profitability ratios | **Required** (no key needed) |
| **Wikipedia** (`pandas.read_html`) | Current S&P 500 constituent list & GICS sectors | **Required** (no key needed) |
| **Financial Modeling Prep (FMP)** | Alternative/backup fundamentals, EV/EBITDA, historical financial statements | Optional — needs free API key |
| **SEC EDGAR** | Raw XBRL filings for the most authoritative fundamental data | Optional — no key, but requires a `User-Agent` header |
| **Alpha Vantage** | Backup price/fundamental data source | Optional — needs free API key |
| **FRED** | Risk-free rate (for Sharpe/Sortino) and macro regime context | Optional — needs free API key |

The core pipeline runs entirely on **`yfinance` + Wikipedia**, which require no API keys, so the notebook runs out-of-the-box in Google Colab. The optional sources are implemented as **pluggable functions** that activate automatically if you paste an API key into the `CONFIG` cell — the pipeline gracefully falls back to `yfinance` otherwise.


In [ ]:
# ==============================================================================
# 6. REQUIRED PYTHON LIBRARIES — INSTALLATION
# ==============================================================================
# yfinance          -> free market data (prices, fundamentals) from Yahoo Finance
# pandas / numpy    -> data wrangling & vectorized numerical computation
# scipy             -> statistics (z-scores, regression for beta/alpha)
# plotly            -> interactive, presentation-quality charts for the dashboard
# matplotlib/seaborn-> static publication-quality charts (heatmaps, correlation)
# ipywidgets        -> interactive controls for the dashboard section
# requests          -> HTTP calls to optional APIs (FMP, SEC EDGAR, Alpha Vantage, FRED)
# tqdm              -> progress bars during bulk data downloads
# lxml / html5lib   -> backends needed by pandas.read_html to scrape Wikipedia

!pip install -q yfinance pandas numpy scipy plotly matplotlib seaborn ipywidgets requests tqdm lxml html5lib --upgrade
print("✅ All libraries installed.")


## 7. Recommended Project Structure

Even though this all lives in one Colab notebook, it is organized internally as if it were a professional GitHub repository:

```
multi-factor-equity-screener/
│
├── README.md                     <- Project overview (generated in Section 27)
├── multi_factor_screener.ipynb   <- This notebook
├── requirements.txt              <- Pinned dependency versions
│
├── src/
│   ├── data_collection.py        <- get_sp500_universe(), fetch_prices(), fetch_fundamentals()
│   ├── data_cleaning.py          <- clean_fundamentals(), handle_missing()
│   ├── factors.py                <- build_value(), build_momentum(), build_quality(), ...
│   ├── scoring.py                <- zscore_frame(), composite_score(), rank_universe()
│   ├── portfolio.py               <- FactorPortfolio class, rebalancing logic
│   ├── performance.py             <- PerformanceAnalyzer class (Sharpe, Sortino, alpha, beta...)
│   └── visuals.py                 <- all Plotly/Matplotlib chart builders
│
├── data/
│   ├── raw/                       <- cached raw pulls (prices.csv, fundamentals.csv)
│   └── processed/                 <- factor_scores.csv, portfolio_holdings.csv
│
└── outputs/
    ├── figures/                   <- exported chart images
    └── tearsheet.html              <- exported interactive dashboard
```

Inside the notebook, each of the modules above is implemented as a clearly labeled section with its functions/classes defined in one cell — mirroring how you'd split real `.py` files in a repo.


## 8. Complete Step-by-Step Development Guide

1. **Configure** — set universe size, lookback window, rebalance frequency, and optional API keys.
2. **Collect** — scrape the S&P 500 list from Wikipedia; download OHLCV price history and fundamental snapshots from `yfinance`.
3. **Clean** — drop tickers with insufficient history, impute or drop missing fundamentals, flag delisted names.
4. **Engineer features** — compute the raw metrics behind each of the five factors.
5. **Score** — convert raw metrics to cross-sectional Z-scores (winsorized to control outliers), then average into per-factor scores and a composite score.
6. **Rank** — sort the universe by composite score; select the top decile (or top N) as the long book.
7. **Construct portfolio** — equal-weight (or score-weighted) the selected names; simulate periodic rebalancing.
8. **Benchmark** — pull SPY as the passive benchmark over the same window.
9. **Analyze performance** — compute return, risk, and risk-adjusted metrics for the factor portfolio vs. SPY.
10. **Visualize** — render the full interactive analytics dashboard.
11. **Package** — auto-generate the README, resume bullets, and LinkedIn post text from the run's actual results.


## 9. Configuration

All tunable parameters live in one place. Change `UNIVERSE_SIZE` to `None` to run the *entire* S&P 500 (slower); leave it at a smaller number (e.g. 75) for a fast, reliable demo run. Optional API keys can be pasted in — leave blank to skip that source.


In [ ]:
# ==============================================================================
# GLOBAL CONFIGURATION
# ==============================================================================
import warnings
warnings.filterwarnings("ignore")

CONFIG = {
    # --- Universe & data window ---
    "UNIVERSE_SIZE": 75,          # None = full S&P 500 (~500 tickers, slow). Try 50-100 for a fast demo.
    "PRICE_LOOKBACK_YEARS": 3,     # years of daily price history to pull
    "BENCHMARK": "SPY",

    # --- Factor / portfolio parameters ---
    "TOP_PCT": 0.10,               # top decile (10%) selected into the long portfolio
    "WEIGHTING_SCHEME": "equal",   # "equal" or "score_weighted"
    "REBALANCE_FREQ": "M",         # 'M' = monthly, 'Q' = quarterly

    # --- Risk-free rate for Sharpe/Sortino (annualized, e.g. 0.04 = 4%) ---
    "RISK_FREE_RATE": 0.04,

    # --- Optional API keys (leave blank string to skip) ---
    "FMP_API_KEY": "",
    "ALPHA_VANTAGE_API_KEY": "",
    "FRED_API_KEY": "",
    "SEC_USER_AGENT": "research-project contact@example.com",

    # --- Reproducibility ---
    "RANDOM_SEED": 42,
}

import numpy as np
np.random.seed(CONFIG["RANDOM_SEED"])

print("Configuration loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


In [ ]:
# ==============================================================================
# CORE IMPORTS
# ==============================================================================
import time
import io
import json
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats
from datetime import datetime, timedelta
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("✅ Imports successful.")


## 10. Data Collection Pipeline

**Step 1 — Universe:** We scrape the live S&P 500 constituent table from Wikipedia (ticker, company name, GICS sector). This is far more current than any hardcoded list.

**Step 2 — Prices:** We bulk-download daily adjusted close prices for the whole universe plus the benchmark via `yfinance.download`, which batches the request efficiently instead of looping ticker-by-ticker.

**Step 3 — Fundamentals:** For each ticker we pull a snapshot of valuation/profitability/leverage metrics from `yfinance.Ticker(...).info`. This *does* require one request per ticker (Yahoo does not offer a bulk fundamentals endpoint), so we add a small delay and a progress bar to stay within reasonable rate limits, and cache results so re-runs are instant.

An optional `fetch_fmp_fundamentals()` function is included that will be used instead/in addition if you supply an FMP API key in `CONFIG`.


In [ ]:
# ==============================================================================
# 10a. UNIVERSE COLLECTION — S&P 500 CONSTITUENTS (Wikipedia)
# ==============================================================================
def get_sp500_universe(universe_size=None):
    \"\"\"
    Scrape the current S&P 500 constituent list from Wikipedia.

    Returns
    -------
    pd.DataFrame with columns: ['ticker', 'company', 'sector']
    \"\"\"
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    try:
        tables = pd.read_html(url)
        raw = tables[0]
        df = raw.rename(columns={
            "Symbol": "ticker",
            "Security": "company",
            "GICS Sector": "sector",
        })[["ticker", "company", "sector"]]
        df["ticker"] = df["ticker"].str.replace(".", "-", regex=False)  # yfinance format (e.g. BRK.B -> BRK-B)
    except Exception as e:
        print(f"⚠️  Could not scrape Wikipedia ({e}). Falling back to a hardcoded mega-cap sample.")
        fallback = ["AAPL","MSFT","AMZN","NVDA","GOOGL","META","BRK-B","LLY","AVGO","JPM",
                    "TSLA","V","XOM","UNH","MA","PG","JNJ","HD","COST","MRK"]
        df = pd.DataFrame({"ticker": fallback, "company": fallback, "sector": "Unknown"})

    if universe_size is not None:
        df = df.head(universe_size).reset_index(drop=True)
    return df.reset_index(drop=True)

universe_df = get_sp500_universe(CONFIG["UNIVERSE_SIZE"])
print(f"Universe size: {len(universe_df)} tickers")
universe_df.head(10)


In [ ]:
# ==============================================================================
# 10b. PRICE HISTORY COLLECTION (batched, vectorized)
# ==============================================================================
def fetch_price_history(tickers, benchmark, years=3):
    \"\"\"
    Bulk-download daily adjusted close prices for the universe + benchmark.

    Returns
    -------
    pd.DataFrame of adjusted close prices, columns = tickers, index = dates.
    \"\"\"
    end = datetime.today()
    start = end - timedelta(days=int(365.25 * years) + 10)
    all_tickers = list(dict.fromkeys(tickers + [benchmark]))  # de-dup, preserve order

    print(f"Downloading {len(all_tickers)} tickers from {start.date()} to {end.date()} ...")
    raw = yf.download(
        all_tickers, start=start, end=end, auto_adjust=True,
        progress=False, group_by="ticker", threads=True
    )

    # yfinance returns a MultiIndex (ticker, field) frame when >1 ticker requested
    px_close = pd.DataFrame({t: raw[t]["Close"] for t in all_tickers if t in raw.columns.get_level_values(0)})
    px_close = px_close.sort_index()
    return px_close

prices = fetch_price_history(universe_df["ticker"].tolist(), CONFIG["BENCHMARK"], CONFIG["PRICE_LOOKBACK_YEARS"])
print(f"Price matrix shape: {prices.shape}")
prices.tail()


In [ ]:
# ==============================================================================
# 10c. FUNDAMENTAL DATA COLLECTION (per-ticker snapshot via yfinance.Ticker.info)
# ==============================================================================
FUNDAMENTAL_FIELDS = {
    "trailingPE": "pe_ratio",
    "priceToBook": "pb_ratio",
    "enterpriseToEbitda": "ev_ebitda",
    "returnOnEquity": "roe",
    "profitMargins": "profit_margin",
    "debtToEquity": "debt_to_equity",
    "marketCap": "market_cap",
    "sector": "sector_yf",
}

def fetch_fundamentals_yf(tickers, pause=0.15):
    \"\"\"
    Pull a fundamentals snapshot per ticker from yfinance's `.info` dict.
    Includes basic retry logic and a progress bar; gracefully skips tickers
    whose data is unavailable (delisted, no coverage, etc.).
    \"\"\"
    records = []
    for t in tqdm(tickers, desc="Fetching fundamentals"):
        row = {"ticker": t}
        for attempt in range(2):
            try:
                info = yf.Ticker(t).get_info()
                for src, dst in FUNDAMENTAL_FIELDS.items():
                    row[dst] = info.get(src, np.nan)
                break
            except Exception:
                time.sleep(0.5)
                if attempt == 1:
                    for dst in FUNDAMENTAL_FIELDS.values():
                        row[dst] = np.nan
        records.append(row)
        time.sleep(pause)
    return pd.DataFrame(records)

def fetch_fmp_fundamentals(tickers, api_key):
    \"\"\"
    OPTIONAL: pull key-metrics from Financial Modeling Prep as an alternative /
    supplemental fundamentals source. Only runs if an API key is supplied.
    \"\"\"
    if not api_key:
        return None
    records = []
    for t in tqdm(tickers, desc="Fetching FMP fundamentals"):
        try:
            url = f"https://financialmodelingprep.com/api/v3/key-metrics-ttm/{t}?apikey={api_key}"
            resp = requests.get(url, timeout=10).json()
            if resp:
                m = resp[0]
                records.append({
                    "ticker": t,
                    "pe_ratio_fmp": m.get("peRatioTTM"),
                    "pb_ratio_fmp": m.get("pbRatioTTM"),
                    "ev_ebitda_fmp": m.get("enterpriseValueOverEBITDATTM"),
                    "roe_fmp": m.get("roeTTM"),
                })
        except Exception:
            continue
    return pd.DataFrame(records) if records else None

fundamentals_raw = fetch_fundamentals_yf(universe_df["ticker"].tolist())

# Optional FMP supplement (only activates if a key was provided in CONFIG)
fmp_data = fetch_fmp_fundamentals(universe_df["ticker"].tolist(), CONFIG["FMP_API_KEY"])
if fmp_data is not None:
    fundamentals_raw = fundamentals_raw.merge(fmp_data, on="ticker", how="left")
    # Prefer FMP EV/EBITDA when yfinance is missing it
    fundamentals_raw["ev_ebitda"] = fundamentals_raw["ev_ebitda"].fillna(fundamentals_raw.get("ev_ebitda_fmp"))
    print("✅ Supplemented fundamentals with Financial Modeling Prep data.")
else:
    print("ℹ️  No FMP API key provided — using yfinance fundamentals only.")

fundamentals_raw.head(10)


### Optional Data Sources (SEC EDGAR, Alpha Vantage, FRED)

These are implemented as opt-in plug-ins so the notebook stays fully runnable with zero API keys. They are not required for the core pipeline but show how the architecture extends to additional institutional data sources.


In [ ]:
# ==============================================================================
# OPTIONAL PLUGIN: SEC EDGAR (company facts — no key required, needs User-Agent)
# ==============================================================================
def fetch_sec_company_facts(ticker_to_cik, user_agent):
    \"\"\"
    OPTIONAL — Pull raw XBRL 'company facts' from SEC EDGAR for a mapping of
    {ticker: CIK}. Included to show how authoritative filing-level data would
    be integrated; not required for the core pipeline since it needs a
    ticker->CIK lookup table and per-filing parsing that is out of scope here.
    \"\"\"
    base = "https://data.sec.gov/api/xbrl/companyfacts/CIK{}.json"
    out = {}
    headers = {"User-Agent": user_agent}
    for ticker, cik in ticker_to_cik.items():
        try:
            url = base.format(str(cik).zfill(10))
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code == 200:
                out[ticker] = resp.json()
        except Exception:
            continue
    return out

# ==============================================================================
# OPTIONAL PLUGIN: Alpha Vantage (backup price source)
# ==============================================================================
def fetch_alpha_vantage_prices(ticker, api_key):
    if not api_key:
        return None
    url = ("https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED"
           f"&symbol={ticker}&outputsize=full&apikey={api_key}")
    try:
        resp = requests.get(url, timeout=10).json()
        ts = resp.get("Time Series (Daily)", {})
        s = pd.Series({d: float(v["5. adjusted close"]) for d, v in ts.items()})
        s.index = pd.to_datetime(s.index)
        return s.sort_index()
    except Exception:
        return None

# ==============================================================================
# OPTIONAL PLUGIN: FRED (risk-free rate override, e.g. 3-Month T-Bill DGS3MO)
# ==============================================================================
def fetch_fred_series(series_id, api_key):
    if not api_key:
        return None
    url = (f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}"
           f"&api_key={api_key}&file_type=json")
    try:
        resp = requests.get(url, timeout=10).json()
        obs = resp.get("observations", [])
        s = pd.Series({o["date"]: o["value"] for o in obs})
        s.index = pd.to_datetime(s.index)
        s = pd.to_numeric(s, errors="coerce").dropna()
        return s
    except Exception:
        return None

# Attempt to override the static risk-free rate with a live FRED 3M T-Bill yield
fred_rf = fetch_fred_series("DGS3MO", CONFIG["FRED_API_KEY"])
if fred_rf is not None and not fred_rf.empty:
    CONFIG["RISK_FREE_RATE"] = float(fred_rf.iloc[-1]) / 100.0
    print(f"✅ Risk-free rate updated from FRED: {CONFIG['RISK_FREE_RATE']:.4%}")
else:
    print(f"ℹ️  Using static risk-free rate from CONFIG: {CONFIG['RISK_FREE_RATE']:.4%}")


## 11. Data Cleaning Pipeline

Real-world financial data is messy: newly-listed companies lack 12-month history, some tickers have no analyst coverage (missing ratios), and REITs/financials often report `NaN` for metrics like `profitMargins`. Our cleaning strategy:

1. **Price panel** — drop tickers missing more than 5% of trading days in the lookback window (illiquid/delisted); forward-fill isolated 1–2 day gaps (holidays/data glitches) but never fill long gaps.
2. **Fundamentals panel** — drop tickers missing *all* factor inputs; for individual missing ratios, leave as `NaN` and let the Z-score step exclude that observation from that specific factor's cross-sectional mean/std (rather than dropping the whole company).
3. **Outlier control** — winsorize each raw metric at the 1st/99th percentile before Z-scoring, so a single data error (e.g., a P/E of 4,000 from a near-zero earnings quarter) can't dominate the ranking.


In [ ]:
# ==============================================================================
# 11. DATA CLEANING
# ==============================================================================
def clean_price_panel(prices, max_missing_pct=0.05):
    \"\"\"Drop tickers with too much missing price history; fill small gaps.\"\"\"
    missing_pct = prices.isna().mean()
    keep = missing_pct[missing_pct <= max_missing_pct].index
    cleaned = prices[keep].ffill(limit=3)
    dropped = sorted(set(prices.columns) - set(keep))
    if dropped:
        print(f"Dropped {len(dropped)} tickers for insufficient price history: {dropped}")
    return cleaned

def clean_fundamentals(fund_df, required_min_fields=2):
    \"\"\"Drop rows where almost no fundamental data is available at all.\"\"\"
    factor_cols = ["pe_ratio", "pb_ratio", "ev_ebitda", "roe", "profit_margin",
                   "debt_to_equity", "market_cap"]
    present = fund_df[factor_cols].notna().sum(axis=1)
    cleaned = fund_df[present >= required_min_fields].copy()
    dropped = set(fund_df["ticker"]) - set(cleaned["ticker"])
    if dropped:
        print(f"Dropped {len(dropped)} tickers for insufficient fundamental coverage: {sorted(dropped)}")
    return cleaned.reset_index(drop=True)

def winsorize(series, lower=0.01, upper=0.99):
    \"\"\"Cap extreme outliers at the given percentiles (leaves NaNs untouched).\"\"\"
    lo, hi = series.quantile(lower), series.quantile(upper)
    return series.clip(lower=lo, upper=hi)

prices_clean = clean_price_panel(prices)
fundamentals_clean = clean_fundamentals(fundamentals_raw)

# Align universe to tickers that survived BOTH the price and fundamentals cleaning
valid_tickers = sorted(set(prices_clean.columns) & set(fundamentals_clean["ticker"]) - {CONFIG["BENCHMARK"]})
fundamentals_clean = fundamentals_clean[fundamentals_clean["ticker"].isin(valid_tickers)].reset_index(drop=True)

print(f"\\nFinal clean universe: {len(valid_tickers)} tickers")
fundamentals_clean.describe()


## 12. Feature Engineering & Factor Construction

Each factor is built from 2–3 raw metrics. **Lower is better** for Value and Volatility metrics (cheap and stable is good), so those raw metrics are *inverted or sign-flipped* before Z-scoring — this is called **factor orientation**, and it ensures that a higher Z-score always means "more attractive" across every factor.

### Value Factor
- **P/E (Price / Earnings)** — price paid per dollar of annual earnings. Lower = cheaper.
- **P/B (Price / Book)** — price paid per dollar of net asset (book) value. Lower = cheaper.
- **EV/EBITDA** — enterprise value per dollar of operating cash earnings; capital-structure-neutral valuation multiple. Lower = cheaper.

### Momentum Factor
- **3-Month, 6-Month, 12-Month Total Return** — trailing price return over each horizon. Higher = stronger momentum. (Institutional desks often skip the most recent month to avoid short-term reversal — noted as an upgrade in Section 30.)

### Quality Factor
- **ROE (Return on Equity)** — net income / shareholder equity; how efficiently the company compounds shareholders' capital. Higher = better.
- **Profit Margin** — net income / revenue; pricing power and cost discipline. Higher = better.
- **Debt-to-Equity** — leverage ratio. Lower = safer balance sheet, so this metric is *inverted*.

### Volatility Factor (Low-Volatility tilt)
- **Annualized Volatility** — standard deviation of daily returns × √252. Lower = more stable, so inverted.
- **Rolling 3-Month Volatility** — a shorter-horizon read on recent risk, also inverted.

### Size Factor
- **Market Capitalization** — institutional size-factor research (Fama-French SMB) finds smaller caps carry a long-run premium, so raw market cap is Z-scored and then **inverted** (smaller = higher factor score) to express a small-cap tilt. (This is configurable — see the `SIZE_ORIENTATION` flag below.)


In [ ]:
# ==============================================================================
# 12a. VALUE FACTOR
# ==============================================================================
def build_value_features(fund_df):
    df = fund_df.copy()
    df["pe_ratio"] = winsorize(df["pe_ratio"].where(df["pe_ratio"] > 0))     # negative P/E is meaningless -> NaN
    df["pb_ratio"] = winsorize(df["pb_ratio"].where(df["pb_ratio"] > 0))
    df["ev_ebitda"] = winsorize(df["ev_ebitda"].where(df["ev_ebitda"] > 0))
    return df

# ==============================================================================
# 12b. MOMENTUM FACTOR (built from the cleaned price panel)
# ==============================================================================
def build_momentum_features(prices, tickers, ref_date=None):
    \"\"\"Compute trailing 3/6/12-month total returns per ticker.\"\"\"
    px = prices[tickers].dropna(how="all")
    ref_date = ref_date or px.index[-1]
    last_price = px.loc[:ref_date].iloc[-1]

    def trailing_return(months):
        target_date = ref_date - pd.DateOffset(months=months)
        past_price = px.loc[:target_date].iloc[-1] if len(px.loc[:target_date]) else np.nan
        return (last_price / past_price) - 1.0

    mom = pd.DataFrame({
        "ticker": tickers,
        "ret_3m": trailing_return(3).reindex(tickers).values,
        "ret_6m": trailing_return(6).reindex(tickers).values,
        "ret_12m": trailing_return(12).reindex(tickers).values,
    })
    return mom

# ==============================================================================
# 12c. QUALITY FACTOR
# ==============================================================================
def build_quality_features(fund_df):
    df = fund_df.copy()
    df["roe"] = winsorize(df["roe"])
    df["profit_margin"] = winsorize(df["profit_margin"])
    df["debt_to_equity"] = winsorize(df["debt_to_equity"].where(df["debt_to_equity"] >= 0))
    return df

# ==============================================================================
# 12d. VOLATILITY FACTOR (built from the cleaned price panel)
# ==============================================================================
def build_volatility_features(prices, tickers, ref_date=None):
    px = prices[tickers].dropna(how="all")
    ref_date = ref_date or px.index[-1]
    daily_ret = px.pct_change()

    ann_vol = daily_ret.loc[:ref_date].std() * np.sqrt(252)
    rolling_vol_3m = daily_ret.loc[:ref_date].tail(63).std() * np.sqrt(252)  # ~63 trading days = 3 months

    vol_df = pd.DataFrame({
        "ticker": tickers,
        "ann_volatility": ann_vol.reindex(tickers).values,
        "rolling_vol_3m": rolling_vol_3m.reindex(tickers).values,
    })
    return vol_df

# ==============================================================================
# 12e. SIZE FACTOR
# ==============================================================================
def build_size_features(fund_df):
    df = fund_df.copy()
    df["market_cap"] = winsorize(df["market_cap"].where(df["market_cap"] > 0))
    return df

# --- Assemble the full feature table ---
fund_value = build_value_features(fundamentals_clean)
fund_quality = build_quality_features(fund_value)
fund_size = build_size_features(fund_quality)

mom_features = build_momentum_features(prices_clean, valid_tickers)
vol_features = build_volatility_features(prices_clean, valid_tickers)

features = (fund_size
            .merge(mom_features, on="ticker", how="left")
            .merge(vol_features, on="ticker", how="left"))

print(f"Feature table shape: {features.shape}")
features.head(10)


## 13–15. Z-Score Normalization, Composite Scoring & Ranking

### Why Z-Scores?
Raw factor metrics live on completely different scales — P/E is a small multiple (e.g. 20), market cap is in the billions, ROE is a percentage. You cannot average them directly. The **Z-score** standardizes each metric to "how many standard deviations above/below the cross-sectional average is this stock," putting every metric on the same unitless scale:

$$ Z_i = \frac{x_i - \mu}{\sigma} $$

where $x_i$ is the stock's raw metric value, $\mu$ is the universe's mean for that metric on that date, and $\sigma$ is the universe's standard deviation. A stock with $Z = 1.5$ on Value is 1.5 standard deviations *cheaper* than the average stock in the universe.

**Sign orientation:** for metrics where *lower is better* (P/E, P/B, EV/EBITDA, Debt/Equity, Volatility), we compute the Z-score and then multiply by −1, so that a higher final factor score always means "more attractive" for every factor.

### Composite Factor Scoring
Each factor's score is the **average of its component Z-scores** (this is itself a form of noise reduction — averaging several noisy signals produces a more stable estimate than any single one). The **Composite Score** is then an equal-weighted average across the five factors:

$$ \text{Composite}_i = \frac{1}{5}\left(Z^{Value}_i + Z^{Momentum}_i + Z^{Quality}_i + Z^{LowVol}_i + Z^{Size}_i\right) $$

Equal weighting across factors is the simplest, most transparent institutional convention (used as a starting baseline before moving to risk-model-based factor weighting — see Section 30).

### Ranking Methodology
Stocks are sorted **descending** by Composite Score. The **top decile** (`TOP_PCT` in `CONFIG`, default 10%) becomes the long portfolio's candidate list.


In [ ]:
# ==============================================================================
# 13. Z-SCORE NORMALIZATION
# ==============================================================================
def zscore(series):
    \"\"\"Cross-sectional Z-score; NaNs are ignored in the mean/std and stay NaN in output.\"\"\"
    return (series - series.mean()) / series.std(ddof=0)

# ==============================================================================
# 14. COMPOSITE FACTOR SCORING
# ==============================================================================
def compute_factor_scores(features):
    df = features.copy()

    # --- Value (lower raw metric = better -> invert Z) ---
    df["z_pe"]  = -zscore(df["pe_ratio"])
    df["z_pb"]  = -zscore(df["pb_ratio"])
    df["z_ev_ebitda"] = -zscore(df["ev_ebitda"])
    df["Value"] = df[["z_pe", "z_pb", "z_ev_ebitda"]].mean(axis=1, skipna=True)

    # --- Momentum (higher = better) ---
    df["z_ret_3m"]  = zscore(df["ret_3m"])
    df["z_ret_6m"]  = zscore(df["ret_6m"])
    df["z_ret_12m"] = zscore(df["ret_12m"])
    df["Momentum"] = df[["z_ret_3m", "z_ret_6m", "z_ret_12m"]].mean(axis=1, skipna=True)

    # --- Quality (ROE, margin higher=better; leverage lower=better -> invert) ---
    df["z_roe"] = zscore(df["roe"])
    df["z_margin"] = zscore(df["profit_margin"])
    df["z_leverage"] = -zscore(df["debt_to_equity"])
    df["Quality"] = df[["z_roe", "z_margin", "z_leverage"]].mean(axis=1, skipna=True)

    # --- Volatility (lower = better -> invert; renamed LowVol for clarity) ---
    df["z_ann_vol"] = -zscore(df["ann_volatility"])
    df["z_roll_vol"] = -zscore(df["rolling_vol_3m"])
    df["LowVol"] = df[["z_ann_vol", "z_roll_vol"]].mean(axis=1, skipna=True)

    # --- Size (small-cap tilt: invert market-cap Z. Flip the sign to go large-cap tilt instead.) ---
    SIZE_ORIENTATION = -1   # -1 = small-cap tilt (Fama-French SMB style), +1 = large-cap tilt
    df["Size"] = SIZE_ORIENTATION * zscore(df["market_cap"])

    # --- Composite: equal-weighted average of the five factor scores ---
    factor_cols = ["Value", "Momentum", "Quality", "LowVol", "Size"]
    df["Composite"] = df[factor_cols].mean(axis=1, skipna=True)

    return df

scored = compute_factor_scores(features)
scored = scored.merge(universe_df[["ticker", "company", "sector"]], on="ticker", how="left")

# ==============================================================================
# 15. RANKING
# ==============================================================================
scored = scored.sort_values("Composite", ascending=False).reset_index(drop=True)
scored["Rank"] = scored.index + 1

n_select = max(1, int(np.ceil(len(scored) * CONFIG["TOP_PCT"])))
top_stocks = scored.head(n_select).copy()

print(f"Ranked universe: {len(scored)} stocks. Selecting top {n_select} ({CONFIG['TOP_PCT']:.0%}).")
scored[["Rank", "ticker", "company", "sector", "Value", "Momentum", "Quality", "LowVol", "Size", "Composite"]].head(15)


## 16. Portfolio Construction

Two weighting schemes are supported (toggle via `CONFIG["WEIGHTING_SCHEME"]`):

- **Equal Weight** — every selected stock gets $1/N$ of the portfolio. Simple, robust, and avoids concentrating risk in whichever stock happens to have the single highest score.
- **Score-Weighted** — weight is proportional to each stock's Composite Z-score (shifted to be positive), tilting more capital toward the highest-conviction names.

**Rebalancing Logic:** the portfolio is reconstituted on the frequency set in `CONFIG["REBALANCE_FREQ"]` ('M' = monthly, 'Q' = quarterly). At each rebalance date, we recompute Momentum and Volatility (which are time-varying) and re-select/re-weight the top decile, then hold those fixed weights until the next rebalance — exactly how a real systematic strategy operates.


In [ ]:
# ==============================================================================
# 16. PORTFOLIO CONSTRUCTION
# ==============================================================================
def assign_weights(top_df, scheme="equal"):
    df = top_df.copy()
    if scheme == "equal":
        df["weight"] = 1.0 / len(df)
    elif scheme == "score_weighted":
        shifted = df["Composite"] - df["Composite"].min() + 1e-6   # ensure strictly positive
        df["weight"] = shifted / shifted.sum()
    else:
        raise ValueError("WEIGHTING_SCHEME must be 'equal' or 'score_weighted'")
    return df

class FactorPortfolio:
    \"\"\"
    Encapsulates factor-score-driven portfolio construction with periodic
    rebalancing, mirroring how a systematic strategy would be productionized.
    \"\"\"
    def __init__(self, prices, fundamentals_fetcher_fn, universe, config):
        self.prices = prices
        self.universe = universe
        self.config = config
        self.holdings_history = {}   # {rebalance_date: DataFrame[ticker, weight]}

    def rebalance(self, ref_date, features_snapshot):
        scored_snapshot = compute_factor_scores(features_snapshot)
        scored_snapshot = scored_snapshot.sort_values("Composite", ascending=False).reset_index(drop=True)
        n = max(1, int(np.ceil(len(scored_snapshot) * self.config["TOP_PCT"])))
        top = scored_snapshot.head(n)
        weighted = assign_weights(top, self.config["WEIGHTING_SCHEME"])
        self.holdings_history[ref_date] = weighted.set_index("ticker")["weight"]
        return weighted

    def backtest_returns(self, rebalance_dates):
        \"\"\"
        Simulate the portfolio's daily return series by holding each
        rebalance's weights fixed until the next rebalance date.
        \"\"\"
        daily_ret = self.prices.pct_change()
        port_ret = pd.Series(index=daily_ret.index, dtype=float)

        for i, rdate in enumerate(rebalance_dates):
            next_date = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else daily_ret.index[-1]
            weights = self.holdings_history.get(rdate)
            if weights is None or weights.empty:
                continue
            window = daily_ret.loc[rdate:next_date, weights.index.intersection(daily_ret.columns)]
            w = weights.reindex(window.columns).fillna(0)
            port_ret.loc[window.index] = window.mul(w, axis=1).sum(axis=1)

        return port_ret.dropna()

# --- Build monthly rebalance schedule over the price history window ---
rebalance_dates = pd.date_range(prices_clean.index.min(), prices_clean.index.max(),
                                 freq=CONFIG["REBALANCE_FREQ"])
rebalance_dates = [d for d in rebalance_dates if d >= prices_clean.index.min() + pd.DateOffset(months=13)]
# (skip the first 13 months so a full 12M momentum lookback is always available)

portfolio = FactorPortfolio(prices_clean, None, valid_tickers, CONFIG)

print(f"Simulating {len(rebalance_dates)} rebalances "
      f"({CONFIG['REBALANCE_FREQ']}) from {rebalance_dates[0].date()} to {rebalance_dates[-1].date()} ...")

for rdate in tqdm(rebalance_dates, desc="Rebalancing"):
    mom_snap = build_momentum_features(prices_clean, valid_tickers, ref_date=rdate)
    vol_snap = build_volatility_features(prices_clean, valid_tickers, ref_date=rdate)
    # Value/Quality/Size fundamentals are treated as slow-moving and held at their latest snapshot
    snap = (fund_size.merge(mom_snap, on="ticker", how="left")
                      .merge(vol_snap, on="ticker", how="left"))
    portfolio.rebalance(rdate, snap)

factor_portfolio_returns = portfolio.backtest_returns(rebalance_dates)
print(f"\\nBacktested {len(factor_portfolio_returns)} daily returns for the factor portfolio.")


## 17–18. Benchmark Comparison & Performance Metrics

The factor portfolio is compared against **SPY** (S&P 500 ETF) over the identical date range. All metrics below are computed on **daily simple returns** unless noted, and annualized using 252 trading days/year.

| Metric | Formula | Interpretation |
|---|---|---|
| **Total Return** | $\prod(1+r_t) - 1$ | Cumulative growth of $1 invested |
| **CAGR** | $(1+\text{Total Return})^{252/n} - 1$ | Annualized compounded growth rate |
| **Annualized Volatility** | $\sigma_{daily} \times \sqrt{252}$ | Annualized standard deviation of returns |
| **Sharpe Ratio** | $\frac{\bar{r} - r_f}{\sigma}$ (annualized) | Excess return per unit of total risk |
| **Sortino Ratio** | $\frac{\bar{r} - r_f}{\sigma_{downside}}$ | Excess return per unit of *downside* risk only |
| **Max Drawdown** | $\min\left(\frac{V_t}{\max(V_{\le t})} - 1\right)$ | Worst peak-to-trough decline |
| **Beta** | $\frac{\text{Cov}(r_p, r_b)}{\text{Var}(r_b)}$ | Sensitivity to benchmark moves |
| **Alpha** | $\bar{r_p} - \left[r_f + \beta(\bar{r_b} - r_f)\right]$ | Return unexplained by market exposure (CAPM) |
| **Information Ratio** | $\frac{\bar{r_p - r_b}}{\sigma(r_p - r_b)}$ | Active return per unit of tracking-error risk |
| **Win Rate** | $\frac{\#\{r_t > 0\}}{n}$ | % of periods with a positive return |


In [ ]:
# ==============================================================================
# 18. PERFORMANCE METRICS
# ==============================================================================
class PerformanceAnalyzer:
    \"\"\"Computes standard institutional performance & risk metrics for a return series.\"\"\"

    TRADING_DAYS = 252

    def __init__(self, returns, benchmark_returns, risk_free_rate=0.0):
        common_idx = returns.index.intersection(benchmark_returns.index)
        self.r = returns.loc[common_idx]
        self.b = benchmark_returns.loc[common_idx]
        self.rf_daily = risk_free_rate / self.TRADING_DAYS

    def total_return(self, r=None):
        r = self.r if r is None else r
        return (1 + r).prod() - 1

    def cagr(self, r=None):
        r = self.r if r is None else r
        n = len(r)
        if n == 0:
            return np.nan
        return (1 + self.total_return(r)) ** (self.TRADING_DAYS / n) - 1

    def ann_volatility(self, r=None):
        r = self.r if r is None else r
        return r.std() * np.sqrt(self.TRADING_DAYS)

    def sharpe(self, r=None):
        r = self.r if r is None else r
        excess = r - self.rf_daily
        return (excess.mean() / excess.std()) * np.sqrt(self.TRADING_DAYS) if excess.std() > 0 else np.nan

    def sortino(self, r=None):
        r = self.r if r is None else r
        excess = r - self.rf_daily
        downside = excess[excess < 0]
        downside_std = downside.std()
        return (excess.mean() / downside_std) * np.sqrt(self.TRADING_DAYS) if downside_std > 0 else np.nan

    def max_drawdown(self, r=None):
        r = self.r if r is None else r
        cum = (1 + r).cumprod()
        running_max = cum.cummax()
        drawdown = cum / running_max - 1
        return drawdown.min(), drawdown

    def beta_alpha(self):
        cov_matrix = np.cov(self.r, self.b)
        beta = cov_matrix[0, 1] / cov_matrix[1, 1]
        rp_ann = self.r.mean() * self.TRADING_DAYS
        rb_ann = self.b.mean() * self.TRADING_DAYS
        rf_ann = self.rf_daily * self.TRADING_DAYS
        alpha = rp_ann - (rf_ann + beta * (rb_ann - rf_ann))
        return beta, alpha

    def information_ratio(self):
        active = self.r - self.b
        return (active.mean() / active.std()) * np.sqrt(self.TRADING_DAYS) if active.std() > 0 else np.nan

    def win_rate(self, r=None):
        r = self.r if r is None else r
        return (r > 0).mean()

    def summary(self):
        beta, alpha = self.beta_alpha()
        dd, _ = self.max_drawdown()
        return pd.Series({
            "Total Return": self.total_return(),
            "CAGR": self.cagr(),
            "Ann. Volatility": self.ann_volatility(),
            "Sharpe Ratio": self.sharpe(),
            "Sortino Ratio": self.sortino(),
            "Max Drawdown": dd,
            "Beta": beta,
            "Alpha (ann.)": alpha,
            "Information Ratio": self.information_ratio(),
            "Win Rate": self.win_rate(),
        })

benchmark_returns = prices_clean[CONFIG["BENCHMARK"]].pct_change().dropna()

analyzer = PerformanceAnalyzer(factor_portfolio_returns, benchmark_returns, CONFIG["RISK_FREE_RATE"])
benchmark_analyzer = PerformanceAnalyzer(benchmark_returns, benchmark_returns, CONFIG["RISK_FREE_RATE"])

perf_table = pd.DataFrame({
    "Factor Portfolio": analyzer.summary(),
    CONFIG["BENCHMARK"]: benchmark_analyzer.summary(),
})
perf_table.loc["Beta", CONFIG["BENCHMARK"]] = 1.0
perf_table.loc["Alpha (ann.)", CONFIG["BENCHMARK"]] = 0.0
perf_table.loc["Information Ratio", CONFIG["BENCHMARK"]] = np.nan

print("=== PERFORMANCE SUMMARY ===")
perf_table


## 19. Professional Visualizations

The cells below build the full chart suite: top-ranked stocks, factor heatmap, allocation, cumulative performance vs. benchmark, rolling return/volatility, drawdown, correlation matrix, factor distributions, and a ranking dashboard.


In [ ]:
# ==============================================================================
# 19a. TOP RANKED STOCKS — bar chart
# ==============================================================================
fig = px.bar(
    top_stocks.sort_values("Composite"),
    x="Composite", y="ticker", orientation="h", color="Composite",
    color_continuous_scale="RdYlGn",
    hover_data=["company", "sector", "Value", "Momentum", "Quality", "LowVol", "Size"],
    title=f"Top {len(top_stocks)} Ranked Stocks by Composite Factor Score",
)
fig.update_layout(height=max(400, 20 * len(top_stocks)), yaxis_title="", xaxis_title="Composite Z-Score")
fig.show()


In [ ]:
# ==============================================================================
# 19b. FACTOR SCORE HEATMAP (top-ranked names)
# ==============================================================================
heatmap_data = top_stocks.set_index("ticker")[["Value", "Momentum", "Quality", "LowVol", "Size", "Composite"]]

fig, ax = plt.subplots(figsize=(8, max(6, 0.35 * len(heatmap_data))))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            linewidths=0.5, cbar_kws={"label": "Z-Score"}, ax=ax)
ax.set_title("Factor Score Heatmap — Top Ranked Stocks", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# 19c. PORTFOLIO ALLOCATION — pie / treemap
# ==============================================================================
latest_weights = assign_weights(top_stocks, CONFIG["WEIGHTING_SCHEME"])

fig = px.treemap(
    latest_weights, path=["sector", "ticker"], values="weight",
    color="Composite", color_continuous_scale="RdYlGn",
    title="Factor Portfolio Allocation (Latest Rebalance) — by Sector & Ticker",
)
fig.update_traces(textinfo="label+percent parent")
fig.show()


In [ ]:
# ==============================================================================
# 19d. PORTFOLIO PERFORMANCE vs BENCHMARK — cumulative growth of $1
# ==============================================================================
cum_port = (1 + factor_portfolio_returns).cumprod()
cum_bench = (1 + benchmark_returns.reindex(factor_portfolio_returns.index)).cumprod()

fig = go.Figure()
fig.add_trace(go.Scatter(x=cum_port.index, y=cum_port.values, name="Factor Portfolio",
                          line=dict(color="#2E86AB", width=2.5)))
fig.add_trace(go.Scatter(x=cum_bench.index, y=cum_bench.values, name=CONFIG["BENCHMARK"],
                          line=dict(color="#888888", width=2, dash="dash")))
fig.update_layout(title="Cumulative Growth of $1 — Factor Portfolio vs. Benchmark",
                   xaxis_title="Date", yaxis_title="Growth of $1", legend_title="Series",
                   hovermode="x unified", height=500)
fig.show()


In [ ]:
# ==============================================================================
# 19e. ROLLING RETURNS & 19f. ROLLING VOLATILITY
# ==============================================================================
roll_window = 63  # ~3 months
rolling_ret_port = factor_portfolio_returns.rolling(roll_window).apply(lambda x: (1 + x).prod() - 1)
rolling_ret_bench = benchmark_returns.reindex(factor_portfolio_returns.index).rolling(roll_window).apply(lambda x: (1 + x).prod() - 1)
rolling_vol_port = factor_portfolio_returns.rolling(roll_window).std() * np.sqrt(252)
rolling_vol_bench = benchmark_returns.reindex(factor_portfolio_returns.index).rolling(roll_window).std() * np.sqrt(252)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=("Rolling 3M Return", "Rolling Annualized Volatility"))
fig.add_trace(go.Scatter(x=rolling_ret_port.index, y=rolling_ret_port, name="Factor Portfolio (Return)",
                          line=dict(color="#2E86AB")), row=1, col=1)
fig.add_trace(go.Scatter(x=rolling_ret_bench.index, y=rolling_ret_bench, name=f"{CONFIG['BENCHMARK']} (Return)",
                          line=dict(color="#888888", dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=rolling_vol_port.index, y=rolling_vol_port, name="Factor Portfolio (Vol)",
                          line=dict(color="#E85D4E")), row=2, col=1)
fig.add_trace(go.Scatter(x=rolling_vol_bench.index, y=rolling_vol_bench, name=f"{CONFIG['BENCHMARK']} (Vol)",
                          line=dict(color="#888888", dash="dash")), row=2, col=1)
fig.update_layout(height=650, title="Rolling Risk & Return Analysis", hovermode="x unified")
fig.show()


In [ ]:
# ==============================================================================
# 19g. DRAWDOWN CHART
# ==============================================================================
_, dd_port = analyzer.max_drawdown()
_, dd_bench = benchmark_analyzer.max_drawdown()

fig = go.Figure()
fig.add_trace(go.Scatter(x=dd_port.index, y=dd_port.values * 100, fill="tozeroy",
                          name="Factor Portfolio", line=dict(color="#E85D4E")))
fig.add_trace(go.Scatter(x=dd_bench.index, y=dd_bench.values * 100, name=CONFIG["BENCHMARK"],
                          line=dict(color="#888888", dash="dash")))
fig.update_layout(title="Drawdown — Factor Portfolio vs. Benchmark",
                   yaxis_title="Drawdown (%)", xaxis_title="Date", hovermode="x unified", height=450)
fig.show()


In [ ]:
# ==============================================================================
# 19h. CORRELATION MATRIX — top-ranked stock returns
# ==============================================================================
top_tickers = top_stocks["ticker"].tolist()
corr = prices_clean[top_tickers].pct_change().dropna().corr()

fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Return Correlation Matrix — Top Ranked Stocks")
fig.update_layout(height=650)
fig.show()


In [ ]:
# ==============================================================================
# 19i. FACTOR DISTRIBUTION — histograms across the full universe
# ==============================================================================
factor_cols = ["Value", "Momentum", "Quality", "LowVol", "Size", "Composite"]
fig = make_subplots(rows=2, cols=3, subplot_titles=factor_cols)
for i, col in enumerate(factor_cols):
    r, c = divmod(i, 3)
    fig.add_trace(go.Histogram(x=scored[col], nbinsx=25, marker_color="#2E86AB", showlegend=False),
                  row=r + 1, col=c + 1)
fig.update_layout(height=550, title="Factor Score Distributions — Full Universe")
fig.show()


In [ ]:
# ==============================================================================
# 19j. STOCK RANKING DASHBOARD — scatter of Composite vs Momentum, sized by market cap
# ==============================================================================
fig = px.scatter(
    scored, x="Value", y="Momentum", size="market_cap", color="Composite",
    color_continuous_scale="RdYlGn", hover_name="ticker",
    hover_data=["company", "sector", "Quality", "LowVol", "Rank"],
    title="Stock Ranking Dashboard — Value vs. Momentum (bubble size = Market Cap)",
)
fig.update_layout(height=600)
fig.show()


## 20. Interactive Dashboard (Widgets)

The cell below adds an `ipywidgets` control panel so a user can interactively change the number of top stocks displayed and the sector filter, live-updating the ranking table and chart without re-running the whole notebook.


In [ ]:
# ==============================================================================
# 20. INTERACTIVE DASHBOARD WITH IPYWIDGETS
# ==============================================================================
import ipywidgets as widgets
from IPython.display import display, clear_output

sector_options = ["All"] + sorted(scored["sector"].dropna().unique().tolist())
sector_dropdown = widgets.Dropdown(options=sector_options, value="All", description="Sector:")
top_n_slider = widgets.IntSlider(value=15, min=5, max=min(50, len(scored)), step=1, description="Top N:")
output = widgets.Output()

def update_dashboard(*args):
    with output:
        clear_output(wait=True)
        df = scored if sector_dropdown.value == "All" else scored[scored["sector"] == sector_dropdown.value]
        df = df.sort_values("Composite", ascending=False).head(top_n_slider.value)

        fig = px.bar(df.sort_values("Composite"), x="Composite", y="ticker", orientation="h",
                     color="Composite", color_continuous_scale="RdYlGn",
                     hover_data=["company", "sector"],
                     title=f"Top {len(df)} Stocks — Sector: {sector_dropdown.value}")
        fig.update_layout(height=max(350, 20 * len(df)))
        fig.show()
        display(df[["Rank", "ticker", "company", "sector", "Composite"]].reset_index(drop=True))

sector_dropdown.observe(update_dashboard, names="value")
top_n_slider.observe(update_dashboard, names="value")

display(widgets.HBox([sector_dropdown, top_n_slider]), output)
update_dashboard()


## 23. Time Complexity & Optimization Notes

- **Universe scraping (Wikipedia):** O(1) single request.
- **Price download (`yfinance.download`, batched):** O(T × K) data points but a single vectorized network call for K tickers, versus O(K) separate calls — this is why we batch rather than loop.
- **Fundamentals (`yfinance.Ticker.info`):** inherently O(K) — Yahoo has no bulk fundamentals endpoint, so this is the pipeline's dominant runtime cost. Mitigations used here: a short `pause` between calls to respect informal rate limits, and (recommended) caching results to CSV so repeat runs are O(1).
- **Z-scoring & ranking:** O(K log K) for the sort, O(K) for the vectorized Z-score math — negligible relative to data collection.
- **Backtest simulation:** O(R × T × K) where R = number of rebalances — vectorized with Pandas rather than Python loops over rows, which is the standard optimization for this class of problem.

## 24. Error Handling

Implemented throughout the pipeline:
- **Missing financial statements / NaNs:** handled by `winsorize()` (NaN-safe) and the factor-score functions using `skipna=True`, so a stock missing one metric still gets scored on its remaining metrics rather than being dropped entirely.
- **API rate limits:** retry-with-backoff in `fetch_fundamentals_yf`, plus a configurable `pause` between requests.
- **Missing tickers / delisted companies:** `clean_price_panel` and `clean_fundamentals` drop names with insufficient coverage and print exactly which tickers were dropped and why (transparency for debugging).
- **Network failures:** every external call (`yfinance`, Wikipedia scrape, FMP/FRED/Alpha Vantage/SEC) is wrapped in `try/except` with a graceful fallback (cached data, skip, or a hardcoded backup list) rather than crashing the whole notebook.

## 25. Best Coding Practices Used In This Notebook

- **PEP 8** naming and formatting throughout.
- **Pure, vectorized functions** (`zscore`, `winsorize`, factor builders) — no manual `for` loops over Pandas rows.
- **Separation of concerns** — collection, cleaning, feature engineering, scoring, portfolio construction, and performance analytics are each isolated in their own functions/classes, mirroring a real `src/` package layout.
- **Config-driven design** — every tunable parameter lives in a single `CONFIG` dict, not scattered magic numbers.
- **Defensive programming** — every network call has explicit error handling and a documented fallback behavior.
- **Docstrings** on every function/class explaining purpose, inputs, and outputs.


In [ ]:
# ==============================================================================
# 26. FINAL PROJECT DELIVERABLES — auto-generated run summary
# ==============================================================================
summary_md = f\"\"\"
### 📊 Run Summary

- **Universe analyzed:** {len(scored)} S&P 500 stocks
- **Portfolio size:** {len(top_stocks)} stocks ({CONFIG['TOP_PCT']:.0%} of universe)
- **Weighting scheme:** {CONFIG['WEIGHTING_SCHEME']}
- **Rebalance frequency:** {CONFIG['REBALANCE_FREQ']}
- **Backtest period:** {factor_portfolio_returns.index.min().date()} to {factor_portfolio_returns.index.max().date()}
- **Factor Portfolio CAGR:** {analyzer.cagr():.2%}  |  **{CONFIG['BENCHMARK']} CAGR:** {benchmark_analyzer.cagr():.2%}
- **Factor Portfolio Sharpe:** {analyzer.sharpe():.2f}  |  **{CONFIG['BENCHMARK']} Sharpe:** {benchmark_analyzer.sharpe():.2f}
- **Alpha (annualized):** {analyzer.beta_alpha()[1]:.2%}
- **Top-ranked stock:** {top_stocks.iloc[0]['ticker']} ({top_stocks.iloc[0]['company']})
\"\"\"
print(summary_md)

# Save processed outputs (mirrors the data/processed/ folder from the repo structure in Section 7)
scored.to_csv("factor_scores.csv", index=False)
top_stocks.to_csv("portfolio_holdings.csv", index=False)
perf_table.to_csv("performance_summary.csv")
print("\\n✅ Saved: factor_scores.csv, portfolio_holdings.csv, performance_summary.csv")


## 27–29. README, Resume, and LinkedIn Description

Full standalone versions of the **GitHub README**, **ATS-friendly resume bullets**, and a **LinkedIn post** are provided as separate files alongside this notebook so they can be copy-pasted directly into a portfolio repo, resume, or post — see `README.md`, `RESUME_BULLETS.md`, and `LINKEDIN_POST.md`.


## 30. Potential Institutional Upgrades

This screener is a transparent, teaching-oriented baseline. Ways to extend it toward a true institutional-grade system:

1. **Barra-style multi-factor risk model** — replace the simple 5-factor composite with a full covariance-based risk model (style factors + industry factors + specific risk).
2. **Risk attribution** — decompose portfolio variance into factor contributions vs. idiosyncratic risk.
3. **Sector-neutral ranking** — Z-score *within* each GICS sector rather than across the whole universe, so the portfolio doesn't just become a sector bet (e.g., all Value picks landing in Energy).
4. **Machine learning ranking** — replace the linear composite with a gradient-boosted or neural ranking model trained to predict forward returns from factor exposures.
5. **PCA factor reduction** — use Principal Component Analysis to discover latent statistical factors and check how much of the variance the five named factors actually explain.
6. **Hidden Markov regime detection** — dynamically detect bull/bear/high-vol regimes and tilt factor weights (e.g., overweight Quality/LowVol in high-vol regimes).
7. **Portfolio optimization (mean-variance / CVaR)** — replace equal/score weighting with a proper optimizer subject to risk and turnover constraints.
8. **Risk parity weighting** — weight positions so each contributes equal risk rather than equal capital.
9. **Black-Litterman allocation** — blend the factor-score "views" with market-implied equilibrium returns for more stable, less extreme weights.
10. **Full multi-factor backtesting engine** — add transaction costs, slippage, turnover constraints, and point-in-time (not look-ahead-biased) fundamental data.
11. **Live screening dashboard** — deploy as a Streamlit/Dash app with a daily cron job refreshing scores, instead of a static notebook run.
